In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.audit;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.audit.pipeline_audit_log
(
    RunID STRING,
    LayerName STRING,
    TableName STRING,
    Status STRING,
    Severity STRING,
    RowsProcessed BIGINT,
    ErrorMessage STRING,
    ExecutionTimestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_lakehouse.audit.pipeline_audit_log
SELECT
    uuid(),
    'Silver',
    'customers',
    'SUCCESS',
    'INFO',
    500,
    NULL,
    CURRENT_TIMESTAMP;

In [0]:
%sql
INSERT INTO retail_lakehouse.audit.pipeline_audit_log
SELECT
    uuid(),
    'Gold',
    'fact_sales',
    'FAILED',
    'HIGH',
    0,
    'Referential integrity failed',
    CURRENT_TIMESTAMP;

In [0]:
%sql
SELECT
    Status,
    Severity,
    COUNT(*) AS TotalIssues
FROM retail_lakehouse.audit.pipeline_audit_log
GROUP BY Status, Severity;

In [0]:
%sql
CREATE OR REPLACE VIEW retail_lakehouse.audit.pipeline_execution_summary AS

SELECT
    COUNT(*) AS TotalChecks,
    SUM(CASE WHEN Status = 'SUCCESS' THEN 1 ELSE 0 END) AS PassedChecks,
    SUM(CASE WHEN Status = 'FAILED' THEN 1 ELSE 0 END) AS FailedChecks,
    SUM(CASE WHEN Severity = 'HIGH' THEN 1 ELSE 0 END) AS HighSeverityIssues,
    MAX(ExecutionTimestamp) AS LastExecutionTime
FROM retail_lakehouse.audit.pipeline_audit_log;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.audit.rejected_sales
USING DELTA
AS

SELECT
    *,
    'FAILED' AS Status,
    'HIGH' AS Severity,
    'Invalid quantity' AS ErrorMessage,
    CURRENT_TIMESTAMP AS AuditTimestamp
FROM retail_lakehouse.bronze.sales
WHERE TRY_CAST(Quantity AS INT) <= 0;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.audit.file_audit_log
(
    FileName STRING,
    ZoneName STRING,
    FileStatus STRING,
    Archived BOOLEAN,
    ProcessedTimestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql
select * from retail_lakehouse.audit.file_audit_log;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.audit.rejected_sales
USING DELTA
AS

SELECT *
FROM retail_lakehouse.bronze.sales
WHERE TRY_CAST(Quantity AS INT) <= 0;